In [47]:
import sys
print(sys.executable)

/Users/upanyachennoju/miniconda3/envs/venv/bin/python


In [69]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
from pathlib import Path
import os
import numpy as np

In [49]:
train_img_dir = Path("../data/raw/train/images")
train_mask_dir = Path("../data/raw/train/masks")

train_image_paths = sorted(list(train_img_dir.glob("*")))
train_mask_paths = sorted(list(train_mask_dir.glob("*")))

print(len(train_image_paths))
print(len(train_mask_paths))

3933
3933


In [50]:
test_img_dir = Path("../data/raw/test/images")
test_mask_dir = Path("../data/raw/test/masks")

test_image_paths = sorted(list(test_img_dir.glob("*")))
test_mask_paths = sorted(list(test_mask_dir.glob("*")))

print(len(test_image_paths))
print(len(test_mask_paths))

860
860


In [51]:
print(train_image_paths[0].name)
print(train_mask_paths[0].name)
print(test_image_paths[0].name)
print(test_mask_paths[0].name)

brisc2025_train_00001_gl_ax_t1.jpg
brisc2025_train_00001_gl_ax_t1.png
brisc2025_test_00001_gl_ax_t1.jpg
brisc2025_test_00001_gl_ax_t1.png


In [52]:
train_transform = A.Compose([
    A.Resize(256, 256),
    A.HorizontalFlip(p=0.5),
    A.Rotate(
        limit=15,
        p=0.5
    ),
    A.RandomBrightnessContrast(
        brightness_limit=0.1,
        contrast_limit=0.1,
        p=0.3
    ),
    A.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    ),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(256, 256),
    A.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    ),
    ToTensorV2()
]) 

In [70]:
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, transforms=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.transforms = transforms

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        mask = Image.open(self.mask_paths[idx])

        if self.transforms:
            augumented = self.transforms(image=np.array(image), mask=np.array(mask))
            image = augumented["image"].float()
            mask = augumented["mask"].unsqueeze(0).float() / 255.0

        return image, mask
    
train_dataset = SegmentationDataset(train_image_paths, train_mask_paths, train_transform)
test_dataset = SegmentationDataset(test_image_paths, test_mask_paths, test_transform)

In [71]:
BATCH_SIZE = 32
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, num_workers=4)

print(f"Dataloaders: {train_dataloader, test_dataloader}") 
print(f"Length of train dataloader: {len(train_dataloader)} batches of {BATCH_SIZE}")
print(f"Length of test dataloader: {len(test_dataloader)} batches of {BATCH_SIZE}")

Dataloaders: (<torch.utils.data.dataloader.DataLoader object at 0x315657a50>, <torch.utils.data.dataloader.DataLoader object at 0x315657950>)
Length of train dataloader: 123 batches of 32
Length of test dataloader: 27 batches of 32


In [72]:
images, masks = next(iter(train_dataloader))

print(images.shape)
print(masks.shape)

torch.Size([32, 3, 256, 256])
torch.Size([32, 1, 256, 256])


In [73]:
print(torch.unique(masks))

tensor([0.0000, 0.0039, 0.0078, 0.0118, 0.0157, 0.0196, 0.0235, 0.0275, 0.9725,
        0.9765, 0.9804, 0.9843, 0.9882, 0.9922, 0.9961, 1.0000])
